# Simulating CBEDs with abTEM

This notebook is used to validate certain aspects of the abTEM simulation, including:
1. How does abTEM set the probe intensity scale
2. Whether CBED intensity changes with increasing thickness, probe position, and whether phonon has any impact of not
3. Check whether single CBED and PACBED behaves differently
4. Check whether the antialias kMax has any effect, and whether

Chia-Hao Lee, 2025.04.29

## Observations
1. The probe scaling of abTEM is default at "making the diffraction intensity sum at 1"
2. The PACBED intensity does change using static potential, with (thickness, tilt_x, tilt_y) = (25, 2, -20). It changes from [1.        , 0.99948144, 0.99851424, 0.9976162] within 27.39 Ang.
3. When I set tilt to 0,0, the intensity changes from [1.        , 0.9994193 , 0.9980344 , 0.99653196] within 27.39 Ang.
4. If I completely remove tilt in the probe argument, the intensity remains [1.        , 0.9994193 , 0.9980344 , 0.99653196] so the tilt is probably implemented correctly.
5. If I simulate (500, None, None), then the final intensity is only 0.90984 after 500 Ang. (Note that we can easily hit OOM on GPU)
6. If I simulate with just a single CBED (`grid_scan.get_positions()[0,0]`) then the intensity can drop significantly [1.0000001  0.86442405 0.7705184  0.68822336 0.6184237  0.55763304 0.5092946  0.46635756 0.43358827 0.4011885  0.37632298 0.375723] in 500 Ang.
7. If I simulate with slightly shifted position (`grid_scan.get_positions()[2,2]`), then the intensity only drops to [1.0000001  0.99348915 0.98345244 0.97695804 0.9715846  0.96527284 0.9580264  0.9526063  0.9469641  0.9422957  0.9355756  0.9354073] in 500 Ang.
8. For single CBED right on top atomic column (`grid_scan.get_positions()[0,0]`), the intensity actually drop less with 5 frozen phonon modes [1.0000001  0.8598266  0.7759857  0.7017316  0.6448931  0.60057086 0.5676053  0.5385068  0.5162346  0.49538374 0.47963178 0.47931552] in 500 Ang.
9. For single CBED between atomic column (`grid_scan.get_positions()[2,2]`), the intensity only drops a little bit with 5 frozen phonon modes [1.0000001  0.99278086 0.98125446 0.9739179  0.9669206  0.95970464 0.9512208  0.9439838  0.9354855  0.9288626  0.9205637  0.9203784 ] in 500 Ang.
10. Going back to PACBED, 100 Ang without phonon would drop to 0.9763166, while with 5 frozen phonon modes it'll drop to 0.9780519.
11. Extend test 10. to 200 Ang without phonon would drop to 0.9562225, while with 5 frozen phonon modes it'll drop to 0.9590652.
12. Changing the `max_angle='cutoff'` to `full` doesn't really get more intensity because it's just increasing the masked region.
13. Double the collection angle to 5 Ang^-1 by halfing the `lateral_sampling = 0.1*2/3` actually recovers the total sum intensity to 0.9948632 when working with 200 Ang, single CBED without phonon at `grid_scan.get_positions()[2,2]`. If it's on the atomic column, it'll still be 0.8545004. Note that the original collection angle of 2.5 Ang^-1 would only give 0.97123516 and 0.61298275 using the same condition.
    
## Conclusions
1. The CBED total intensity does change with thickness, total intensity generally drop with increasing thickness.
2. For single CBED, the intensity drops significantly on atomic columns, while it doesn't change as much when it's not hitting the atomic column.
3. Including frozen phonon modes would have different effects depending whether your probe is on atomic column or on vacuum. Although it has minimal effect to PACBED total intensity, it will still change the k-space intensity distribution significantly.
4. The major reason to lose intensity is just the 2/3 antialias kMax that abTEM enforced # https://abtem.readthedocs.io/en/latest/user_guide/appendix/antialiasing.html

## Implications
1. While the total intensity is sensitive to thickness when there's limited collection angle, experimental data are not on absolute scale so we can't actually utilize that information, unless we acquire and compare with a diffraction pattern taken at vacuum.

In [ ]:
import os
work_dir = "H:\workspace\\bott"
os.chdir(work_dir)
print("Current working dir: ", os.getcwd())

## All the parameters are configured below

In [ ]:
# Input unknown
thickness, tilt_x, tilt_y = 100, 0, 0 #25, 2, -20 #(Ang, mrad, mrad)

# Setup abtem device
device_abtem = 'gpu'

# Read cif into abTEM for 4D-STEM generation
path_crystal = './data/SrTiO3.cif'

# Potential params
potential_extent_x = 62.6 # Ang
potential_extent_y = 62.6 # Ang
lateral_sampling = 0.2*2/3 #0.2*2/3 # Ptycho recon pixel size = 0.2 Ang, so ptycho CBED need 1/2dx = 2.5 Ang-1 kMax. Considering the 2/3 kMax antialias, we need 2.5*1.5 = 3.75 Ang-1 for abTEM CBED or equivalently 0.1333 Ang px
vertical_sampling = 2 # Ang
potential_parametrization = "lobato" # Seems to be more accurate then "kirkland"
potential_projection = "finite" # infinite is faster but less accurate
exit_planes = None # Output diffraction pattern every N slices

# Phonon params
random_seed = 42
use_frozen_phonon = False
num_phonon_configs = 25
phonon_sigma = {'Sr':.088,'Ti':.0746,'O':.0963} #0.1 # Ang 

# Probe params
energy = 200e3
convergence_angle = 19.1
df = 0
aberrations = {}

# Scan and pacbed
scan_step_size = 0.3
return_pacbed = True

## Everything below is just calculation

In [ ]:
# Setup imports
from bott.utils import get_EM_constants

import ase
import abtem
import numpy as np
import dask

if device_abtem == 'gpu':
    import cupy as xp
    abtem.config.set({"dask.chunk-size-gpu" : "2048 MB"})
    dask.config.set({"num_workers": 1});
elif device_abtem == 'cpu':
    import numpy as xp
else:
    raise ValueError(f"device_abtem '{device_abtem}' not implemented yet, please use 'cpu', or 'gpu'!")

# abtem configure
abtem.config.set({"local_diagnostics.progress_bar": False});
abtem.config.set({"device": device_abtem});

import matplotlib.pyplot as plt

In [ ]:
# Setup cell
unit_cell = ase.io.read(path_crystal)
target_object_extent = np.array((potential_extent_x, potential_extent_y, thickness)) # Specimen range in Ang (x,y,z)
cell_constants = np.diag(unit_cell.cell)
super_cell_reps = np.ceil(target_object_extent / cell_constants).astype('int')
super_cell = unit_cell * super_cell_reps
print(f"super_cell = {super_cell}")

In [ ]:
# Calculate the potential
if use_frozen_phonon:
    print(f"Using FrozenPhonons potential with {num_phonon_configs} configs")
    atoms = abtem.FrozenPhonons(atoms=super_cell, num_configs=num_phonon_configs, sigmas=phonon_sigma, seed=random_seed)
    potential = abtem.Potential(atoms=atoms, sampling=lateral_sampling, parametrization=potential_parametrization,
        slice_thickness=vertical_sampling, projection=potential_projection, exit_planes=exit_planes)
    # potential_arr = np.mean(potential.build().compute(progress_bar=False).array.get(), axis=0)
else:
    print("Using Static potential")
    potential = abtem.Potential(atoms=super_cell, sampling=lateral_sampling, parametrization=potential_parametrization,
        slice_thickness=vertical_sampling, projection=potential_projection, exit_planes=exit_planes)
    # potential_arr = potential.build().compute(progress_bar=False).array
    
print(f"potential.shape = {potential.shape}")

In [ ]:
# Calculate the probe
# probe = abtem.Probe(energy=energy, semiangle_cutoff=convergence_angle, defocus=df, tilt=(tilt_x,tilt_y), **aberrations)
probe = abtem.Probe(energy=energy, semiangle_cutoff=convergence_angle, defocus=df, **aberrations)
probe.grid.match(potential)

# Useful information
wavelength = get_EM_constants(energy/1e3, 'wavelength')
kmax_antialias = 1/lateral_sampling/3 # 1/Ang #The kmax_antialiasing = 2.675 Ang-1 
alpha_max_antialias = wavelength * kmax_antialias # rad
print(f"Energy = {energy/1e3} kV, rel. wavelength = {wavelength:.4g} Ang")
print(f"CBED collection kmax = {kmax_antialias:.4g} 1/Ang, collection alpha_max = {alpha_max_antialias*1000:.4g} mrad")
print(f"probe.shape = {probe.shape}")
print(f"probe.axes_metadata = {probe.axes_metadata}")

In [ ]:
# Get probe_arr
try:
    probe_arr = probe.build().compute().array.get() # .get() is used for cupy
except AttributeError:
    probe_arr = probe.build().compute().array

In [ ]:
# Check the probe intensity
probe_int = np.abs(probe_arr)**2
print(f"probe_arr.abs (min, mean, max) = ({np.abs(probe_arr).min():.4g},{np.abs(probe_arr).mean():.4g},{np.abs(probe_arr).max():.4g})")
print(f"probe_arr.abs.sum = ({np.abs(probe_arr).sum():.4g})")
print(f"probe_int (min, mean, max) = ({probe_int.min():.4g},{probe_int.mean():.4g},{probe_int.max():.4g})")
print(f"probe_int.sum = ({probe_int.sum():.4g})\n")

# By default the probe normalization is done such that the probe intensity is normalized in k-space
# https://abtem.readthedocs.io/en/latest/reference/api/_autosummary/abtem.waves.Waves.html#abtem.waves.Waves.normalize
probe_arr_fft = np.fft.fft2(probe_arr)
probe_fft_int = np.abs(probe_arr_fft)**2
print(f"probe_arr_fft.abs (min, mean, max) = ({np.abs(probe_arr_fft).min():.4g},{np.abs(probe_arr_fft).mean():.4g},{np.abs(probe_arr_fft).max():.4g})")
print(f"probe_arr_fft.abs.sum = ({np.abs(probe_arr_fft).sum():.4g})")
print(f"probe_fft_int (min, mean, max) = ({probe_fft_int.min():.4g},{probe_fft_int.mean():.4g},{probe_fft_int.max():.4g})")
print(f"probe_fft_int.sum = ({probe_fft_int.sum():.4g})")

In [ ]:
# Make scan for 1 unit cell along x, y
potential_extent = np.array(potential.extent)
scan_start = np.array(potential_extent)/2
scan_end = scan_start + cell_constants[:2]
grid_scan = abtem.scan.GridScan(start=scan_start, end=scan_end,sampling=scan_step_size)
print(f"grid_scan.axes_metadata = {grid_scan.axes_metadata}")
print(grid_scan.shape)

In [ ]:
# # Get cbeds
# cbeds = probe.multislice(scan = grid_scan, potential = potential).diffraction_patterns(max_angle='cutoff').reduce_ensemble().compute(progress_bar=False)
# print(f"cbeds.axes_metadata = {cbeds.axes_metadata}")

# if return_pacbed:
#     measurement = xp.mean(cbeds.array, axis=(-3,-4))
# else:
#     measurement = cbeds.array

# # Cast measurement into numpy array
# if device_abtem == 'gpu':
#     measurement = measurement.get()
# print(f"\nmeasurement.shape = {measurement.shape}")

In [ ]:
# Directly returning the intensity sum of PACBED to reduce the peak memory usage
# measurement_sum = probe.multislice(scan = grid_scan, potential = potential).diffraction_patterns(max_angle='cutoff').reduce_ensemble().mean((-3,-4)).compute().array.sum((-1,-2)) # (num_exit_planes,1)

# Return the intensity sum of a single CBED
# measurement_sum = probe.multislice(scan = grid_scan.get_positions()[2,2], potential = potential).diffraction_patterns(max_angle='cutoff').reduce_ensemble().compute().array.sum((-1,-2)) # (num_exit_planes,1) # Single CBED

measurement = probe.multislice(scan = grid_scan.get_positions()[0,0], potential = potential).diffraction_patterns(max_angle='cutoff').reduce_ensemble().compute().array # (num_exit_planes,1) # Single CBED

# # Cast measurement into numpy array
# if device_abtem == 'gpu':
#     measurement_sum = measurement_sum.get()
# print(measurement_sum)

## Try the function version

In [ ]:
# from bott.physics_models import simulate_cbed
# measurement = simulate_cbed(thickness, tilt_x, tilt_y, device_simu=device_abtem)

# plt.figure()
# plt.imshow(measurement)
# plt.show()

In [ ]:
# measurement.sum()